# 1. Environment Setup
Install the necessary libraries for PDF processing, LangChain, and model execution.

In [1]:
!pip install langchain
!pip install langchain-community
!pip install langchain-openai
!pip install chromadb
!pip install pypdf
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.0 which is inco

In [2]:
pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 56.2 MB/s eta 0:00:00


In [3]:
!pip install transformers accelerate torch

In [4]:
from google.colab import files
uploaded = files.upload()

Saving sample.pdf to sample.pdf


# 2. Document Loading & Preprocessing
Use `pdfplumber` to extract text and metadata from the uploaded PDF, then split it into manageable chunks.

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("sample.pdf")
documents = loader.load()

In [6]:
import pdfplumber
from langchain_core.documents import Document

# Extract text and metadata using pdfplumber
pdf_texts = []
with pdfplumber.open("sample.pdf") as pdf:
    for page_num, page in enumerate(pdf.pages):
        text = page.extract_text()
        if text:
            metadata = {
                'source': 'sample.pdf',
                'page': page_num + 1,
                'producer': pdf.metadata.get('Producer'),
                'creator': pdf.metadata.get('Creator'),
                'title': pdf.metadata.get('Title'),
                'author': pdf.metadata.get('Author')
            }
            pdf_texts.append(Document(page_content=text, metadata=metadata))

documents = pdf_texts


In [7]:
print(documents[1].page_content)

Inthereport,wediscussedrelatedworkin2,statedtheobjectiveofthisprojectandimplementionof
preprocessingstepsalongwiththemodeldesignin3. Wepresentourexperimentin4,resultsin5
andconcludeourreportin7.
2 RelatedWork
TherapidadvancementinmultimodalQAstemsfromintegratingRAGintomulti-datamodality
frameworks. Thissectionreviewsrelevantstudiesanddevelopments,highlightingtheircontributions,
methodologiesandlimitationsofIntegrationofRAGwithPDFProcessingforQA.
2.1 Retrieval-AugmentedGeneration(RAG)
Large language models (LLMs) have advanced AI but have limitations like hallucinations and
inaccuracies. RAGimprovestextaccuracybyleveragingretrieveddocuments. CorrectiveRetrieval
AugmentedGeneration(CRAG)introducesevaluatorstoassessdocumentqualityandrefineretrieval
actions Yan et al. [2024]. Unlike RAG, RAG-end2end Siriwardhana et al. [2023] jointly trains
retrieversandgenerators,enhancingopen-domainquestionansweringbyupdatingallcomponents,
includingexternalknowledgebases.
2.2 QuestionAnswering(QA)withLan

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

# 3. Vector Database
Create embeddings using a HuggingFace model and store the processed documents in a ChromaDB vector store for retrieval.

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_12044/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
print(len(docs))

88


In [12]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings
)

# 4. LLM Initialization
Initialize the Qwen 2.5 model using the HuggingFace transformers pipeline for generating answers.

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# 5. RAG Pipeline (Retrieval & Generation)
Define a query, retrieve relevant context from the vector store, and generate a final answer using the LLM.

In [18]:
retriever = vectorstore.as_retriever()

query = "When the 11 dec airplane crash happened?"

results = retriever.invoke(query)

context = "\n\n".join(
    [doc.page_content for doc in results[:3]]
)

In [21]:
prompt = f"""
You are an AI assistant.

Answer the question only using the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

In [24]:
response = llm(
    prompt,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.5
)

print(response[0]["generated_text"])

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are an AI assistant.

Answer the question only using the provided context.

Context:
VikasParuchur. Insidemarker: Aguidedsourcecodetourforanai-poweredpdflayoutdetection
engine, 2023. URL https://journal.hexmos.com/marker-pdf-document-ai/. Last
visitedDecember10,2023.
Hossein Rajabzadeh, Suyuchen Wang, Hyock Ju Kwon, and Bang Liu. Multimodal multi-hop
questionansweringthroughaconversationbetweentoolsandefficientlyfinetunedlargelanguage
models. arXivpreprintarXiv:2309.08922,2023.
Jon Saad-Falcon, Joe Barrow, Alexa Siu, Ani Nenkova, Ryan A Rossi, and Franck Dernoncourt.

DavideTestuggine,DeliaDavid,DeviParikh,DianaLiskovich,DidemFoss,DingkangWang,Duc
Le,DustinHolland,EdwardDowling,EissaJamil,ElaineMontgomery,EleonoraPresani,Emily
Hahn,EmilyWood,Eric-TuanLe,ErikBrinkman,EstebanArcaute,EvanDunbar,EvanSmothers,
FeiSun,FelixKreuk,FengTian,FilipposKokkinos,FiratOzgenel,FrancescoCaggioni,Frank
Kanayet,FrankSeide,GabrielaMedinaFlorez,GabriellaSchwarz,GadaBadeer,GeorgiaSwee,
GilHalpern,Grant

In [25]:
retriever = vectorstore.as_retriever()

query = "What method was used for PDF preprocessing?"

results = retriever.invoke(query)

context = "\n\n".join(
    [doc.page_content for doc in results[:3]]
)

In [26]:
prompt = f"""
You are an AI assistant.

Answer the question only using the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

In [27]:
response = llm(
    prompt,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.5
)

print(response[0]["generated_text"])

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are an AI assistant.

Answer the question only using the provided context.

Context:
Pdftriage: questionansweringoverlong,structureddocuments. arXivpreprintarXiv:2309.08872,
2023.
Parth Sarthi, Salman Abdullah, Aditi Tuli, Shubh Khanna, Anna Goldie, and Christopher D
Manning. Raptor: Recursiveabstractiveprocessingfortree-organizedretrieval. arXivpreprint
arXiv:2401.18059,2024.
Kunal Sawarkar, Abhilasha Mangal, and Shivam Raj Solanki. Blended rag: Improving rag
(retriever-augmentedgeneration)accuracywithsemanticsearchandhybridquery-basedretrievers.

3.2.1 PDFPreprocessing
Figure2: PDFPreprocessing
Headerandfooterremoval
Theremovalofheadersandfootersisanimportantpreprocessingstepastheycouldinterferewiththe
retrievalprocessbyaddingnoisetothedatawhichleadstolessaccurateresults. Thus,byremoving
headers and footers, we obtain reliable cleaned pdf documents for further processing. We make
anassumptionthatheadersandfooterscoordinatesareconsistentacrosspages,i.e.,atthetopand

3.2 SystemDes